In [1]:
import sys
sys.path.append("..")
from helpers import load_csv_data
import numpy as np

x_train, x_test, y_train, train_ids, test_ids = load_csv_data("../data/", sub_sample=True)

# The first 13 columns (indices 0 to 12) are survey metadata (IDs, dates, etc.)
# drop
cols_to_drop = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] 
x_train_clean = np.delete(x_train, cols_to_drop, axis=1)

print("Original shape:", x_train.shape)
print("Cleaned shape:", x_train_clean.shape)
print("Max value after dropping metadata:", np.max(x_train_clean))

Original shape: (6563, 321)
Cleaned shape: (6563, 308)
Max value after dropping metadata: nan


In [2]:
# find columns have NaNs
nan_mask = np.isnan(x_train_clean)
cols_with_nans = np.where(nan_mask.any(axis=0))[0]

print("Columns with NaNs:", cols_with_nans)

# Look at the first row that has a NaN in the first bad column
if len(cols_with_nans) > 0:
    bad_col = cols_with_nans[0]
    bad_rows = np.where(np.isnan(x_train_clean[:, bad_col]))[0]
    print(f"Column {bad_col} has NaNs in rows: {bad_rows}")
    print("Values in this column:", x_train_clean[:5, bad_col])

Columns with NaNs: [  0   1   2   3   4   5   6   7   8   9  10  11  12  16  22  24  25  28
  36  41  42  43  47  48  49  50  51  52  53  54  55  56  57  58  59  60
  61  62  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78
  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96
  97  98  99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114
 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132
 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150
 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168
 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186
 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 204 205
 207 210 211 212 213 215 221 225 232 237 238 239 240 241 253 254 255 256
 257 258 263 264 272 273 276 277 278 279 280 281 282 283 284 286 287 288
 289 290 291 300 301 302 305 306 307]
Column 0 has NaNs in rows: [   0    2    3 ... 6557 6558 6560]
Valu

In [3]:
# Replace NaNs and Infs with 0 
x_train_clean = np.nan_to_num(x_train_clean, nan=0.0, posinf=0.0, neginf=0.0)

print("NaNs remaining in raw data:", np.isnan(x_train_clean).sum())

NaNs remaining in raw data: 0


In [4]:
from src.preprocessing import standardize, add_bias, to_01

x_train_norm, mean, std = standardize(x_train_clean)
tx_train = add_bias(x_train_norm)
y_train_01 = to_01(y_train)

print("Max value in tx_train:", np.max(tx_train))
print("Min value in tx_train:", np.min(tx_train))

Max value in tx_train: 81.00617260431453
Min value in tx_train: -5.571210522320454


In [18]:
from implementations import logistic_regression
import time

initial_w = np.zeros(tx_train.shape[1])
max_iters = 1000
gamma = 0.1

print("Training logistic regression (No Undersampling)...")
start_time = time.time()
w, loss = logistic_regression(y_train_01, tx_train, initial_w, max_iters, gamma)
end_time = time.time()

print(f"Training took {end_time - start_time:.2f} seconds.")
print(f"Final loss: {loss:.4f}")

predictions = (tx_train @ w > 0).astype(int)
accuracy = np.mean(predictions == y_train_01)
print(f"Training Accuracy: {accuracy * 100:.2f}%")

# Check confusion matrix
tp = np.sum((predictions == 1) & (y_train_01 == 1))
fp = np.sum((predictions == 1) & (y_train_01 == 0))
fn = np.sum((predictions == 0) & (y_train_01 == 1))
tn = np.sum((predictions == 0) & (y_train_01 == 0))
print(f"True Positives: {tp}, False Positives: {fp}, False Negatives: {fn}, True Negatives: {tn}")

# Calculate class imbalance
num_sick = np.sum(y_train_01 == 1)
num_healthy = np.sum(y_train_01 == 0)
total = len(y_train_01)

percentage_sick = (num_sick / total) * 100
percentage_healthy = (num_healthy / total) * 100

print(f"Number of healthy (-1): {num_healthy}")
print(f"Number of sick (+1): {num_sick}")
print(f"Total samples: {total}")
print(f"Imbalance Ratio: {percentage_sick:.2f}% Sick / {percentage_healthy:.2f}% Healthy")

# --- NEW METRICS CALCULATIONS ---
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Training logistic regression (No Undersampling)...
Training took 1.31 seconds.
Final loss: 0.2029
Training Accuracy: 92.17%
True Positives: 103, False Positives: 56, False Negatives: 458, True Negatives: 5946
Number of healthy (-1): 6002
Number of sick (+1): 561
Total samples: 6563
Imbalance Ratio: 8.55% Sick / 91.45% Healthy
Precision: 0.6478
Recall: 0.1836
F1-Score: 0.2861


In [9]:
def build_k_indices(y, k_fold, seed=1):
    """Build k indices for k-fold cross-validation."""
    num_row = y.shape[0]
    interval = int(num_row / k_fold)
    np.random.seed(seed)
    indices = np.random.permutation(num_row)
    k_indices = [indices[k * interval: (k + 1) * interval] for k in range(k_fold)]
    return np.array(k_indices)

def cross_validation(y, x, k_indices, k, lambda_, gamma, max_iters, use_reg=False, balance=False):
    # Get indices
    test_indices = k_indices[k]
    train_indices = np.concatenate([k_indices[j] for j in range(len(k_indices)) if j != k])
    
    # Split data
    x_tr, y_tr = x[train_indices], y[train_indices]
    x_te, y_te = x[test_indices], y[test_indices]
    
    # --- UNDERSAMPLING LOGIC ---
    if balance:
        # Find the minority and majority classes in the TRAIN set
        minority_indices = np.where(y_tr == 1)[0]
        majority_indices = np.where(y_tr == 0)[0]
        
        # Randomly sample majority class to match minority class size
        sampled_majority = np.random.choice(majority_indices, size=len(minority_indices), replace=False)
        
        # Combine and shuffle
        balanced_indices = np.concatenate([minority_indices, sampled_majority])
        np.random.shuffle(balanced_indices)
        
        x_tr = x_tr[balanced_indices]
        y_tr = y_tr[balanced_indices]
    # ---------------------------

    # Standardize
    x_tr_norm, mean, std = standardize(x_tr)
    x_te_norm, _, _ = standardize(x_te, mean, std)
    
    # Add bias
    tx_tr = add_bias(x_tr_norm)
    tx_te = add_bias(x_te_norm)
    
    initial_w = np.zeros(tx_tr.shape[1])
    
    # Train
    if use_reg:
        w, loss = reg_logistic_regression(y_tr, tx_tr, lambda_, initial_w, max_iters, gamma)
    else:
        w, loss = logistic_regression(y_tr, tx_tr, initial_w, max_iters, gamma)
        
    # Predict
    predictions = (tx_te @ w > 0.5).astype(int) 
    
    # Metrics
    accuracy = np.mean(predictions == y_te)
    tp = np.sum((predictions == 1) & (y_te == 1))
    fp = np.sum((predictions == 1) & (y_te == 0))
    fn = np.sum((predictions == 0) & (y_te == 1))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return accuracy, f1, precision, recall

In [10]:
from implementations import logistic_regression
from src.preprocessing import standardize, add_bias, to_01
import numpy as np

# Ensure data is clean
x_train_clean = np.nan_to_num(x_train_clean, nan=0.0, posinf=0.0, neginf=0.0)

k_fold = 5
np.random.seed(42) # Set seed for reproducible sampling
k_indices = build_k_indices(y_train_01, k_fold, seed=1)

accuracies = []
f1s = []
precisions = []
recalls = []

print("5-Fold CV with Undersampling")
for k in range(k_fold):
    acc, f1, prec, rec = cross_validation(y_train_01, x_train_clean, k_indices, k, lambda_=0.1, gamma=0.1, max_iters=1000, use_reg=False, balance=True)
    accuracies.append(acc)
    f1s.append(f1)
    precisions.append(prec)
    recalls.append(rec)
    print(f"Fold {k+1}: Accuracy = {acc*100:.2f}%, F1 = {f1:.4f}, Recall = {rec:.4f}")

print(f"\nMean Accuracy: {np.mean(accuracies)*100:.2f}%")
print(f"Mean F1-Score: {np.mean(f1s):.4f}")
print(f"Mean Recall: {np.mean(recalls):.4f}")

5-Fold CV with Undersampling
Fold 1: Accuracy = 73.55%, F1 = 0.3236, Recall = 0.6803
Fold 2: Accuracy = 75.84%, F1 = 0.3094, Recall = 0.6893
Fold 3: Accuracy = 74.85%, F1 = 0.2795, Recall = 0.6598
Fold 4: Accuracy = 75.61%, F1 = 0.3103, Recall = 0.6486
Fold 5: Accuracy = 75.00%, F1 = 0.3167, Recall = 0.5984

Mean Accuracy: 74.97%
Mean F1-Score: 0.3079
Mean Recall: 0.6553


In [12]:
from implementations import logistic_regression, reg_logistic_regression
lambdas = [0.001, 0.01, 0.1, 1.0, 10.0]
k_fold = 5

print("Sweep lambda for Regularized Logistic Regression\n")

for l in lambdas:
    fold_f1s = []
    fold_accs = []
    
    for k in range(k_fold):
        acc, f1, prec, rec = cross_validation(
            y_train_01, x_train_clean, k_indices, k, 
            lambda_=l, gamma=0.1, max_iters=1000, use_reg=True, balance=True
        )
        fold_f1s.append(f1)
        fold_accs.append(acc)
        
    print(f"Lambda = {l:<6} | Mean Accuracy = {np.mean(fold_accs)*100:.2f}% | Mean F1 = {np.mean(fold_f1s):.4f}")

Sweep lambda for Regularized Logistic Regression

Lambda = 0.001  | Mean Accuracy = 75.58% | Mean F1 = 0.3183
Lambda = 0.01   | Mean Accuracy = 77.90% | Mean F1 = 0.3284
Lambda = 0.1    | Mean Accuracy = 81.57% | Mean F1 = 0.3616
Lambda = 1.0    | Mean Accuracy = 87.27% | Mean F1 = 0.3600
Lambda = 10.0   | Mean Accuracy = 50.29% | Mean F1 = 0.1053


In [13]:
# 1. Clean the test data exactly as we did for the train data
# Drop the same metadata columns (first 13)
x_test_clean = np.delete(x_test, cols_to_drop, axis=1)
x_test_clean = np.nan_to_num(x_test_clean, nan=0.0, posinf=0.0, neginf=0.0)

print("x_test_clean shape:", x_test_clean.shape)

x_test_clean shape: (109379, 308)


In [14]:
from src.preprocessing import standardize, add_bias, to_01, to_pm1
from implementations import reg_logistic_regression
from helpers import create_csv_submission

# 1. Balance the FULL training set (Undersampling)
minority_indices = np.where(y_train_01 == 1)[0]
majority_indices = np.where(y_train_01 == 0)[0]
sampled_majority = np.random.choice(majority_indices, size=len(minority_indices), replace=False)

balanced_indices = np.concatenate([minority_indices, sampled_majority])
np.random.shuffle(balanced_indices)

x_train_bal = x_train_clean[balanced_indices]
y_train_bal = y_train_01[balanced_indices]

# 2. Standardize (fit on balanced train, apply to both train and test)
x_train_bal_norm, mean, std = standardize(x_train_bal)
x_test_norm, _, _ = standardize(x_test_clean, mean, std)

# 3. Add bias
tx_train_bal = add_bias(x_train_bal_norm)
tx_test = add_bias(x_test_norm)

# 4. Train the winning model
initial_w = np.zeros(tx_train_bal.shape[1])
print("Training final model...")
w_final, loss_final = reg_logistic_regression(
    y_train_bal, tx_train_bal, lambda_=1.0, initial_w=initial_w, max_iters=1000, gamma=0.1
)
print("Final training loss:", loss_final)

# 5. Predict on the test set
# (Remember: logistic regression outputs probabilities, we round to 0 or 1 based on 0.5)
test_predictions_01 = (tx_test @ w_final > 0.5).astype(int)

# 6. Convert to {-1, 1} for submission
test_predictions_pm1 = to_pm1(test_predictions_01)

print("Predicted class counts:", np.unique(test_predictions_pm1, return_counts=True))

Training final model...
Final training loss: 0.542125569312567
Predicted class counts: (array([-1,  1]), array([95530, 13849]))


In [15]:
# Create the submission file
create_csv_submission(test_ids, test_predictions_pm1, "submission_v1.csv")
print("Submission saved as submission_v1.csv!")

Submission saved as submission_v1.csv!
